# Modelagem Supervisionada

In [10]:
#importando bibliotecas
import pandas as pd
import plotly.graph_objects as go
import plotly.figure_factory as ff 

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

pd.set_option('display.max_columns', None)

In [11]:
#importando dados
df = pd.read_pickle('../data/curated/features.pkl')
df = df.drop(columns=['DESTINATION_AIRPORT_PROFILE', 'ORIGIN_AIRPORT_PROFILE', 'AIRLINE_PROFILE', 'ROUTE_PROFILE'])
df.head()

,MONTH,DAY_OF_WEEK,SCHEDULED_DEPARTURE_HOUR,SCHEDULED_ARRIVAL_HOUR,SCHEDULED_TIME,DISTANCE,ORIGIN_AWND,ORIGIN_PRCP,ORIGIN_SNOW,ORIGIN_SNWD,ORIGIN_TMAX,ORIGIN_TMIN,ORIGIN_WSF2,ORIGIN_WT01,IS_DELAYED,SEASON_AUTUMN,SEASON_SPRING,SEASON_SUMMER,SEASON_WINTER,TIME_OF_DAY_AFTERNOON,TIME_OF_DAY_EVENING,TIME_OF_DAY_MORNING,TIME_OF_DAY_OVERNIGHT,AIRLINE_AA,AIRLINE_AS,AIRLINE_B6,AIRLINE_DL,AIRLINE_EV,AIRLINE_F9,AIRLINE_HA,AIRLINE_MQ,AIRLINE_NK,AIRLINE_OO,AIRLINE_UA,AIRLINE_US,AIRLINE_VX,AIRLINE_WN
0,-0.654094,1.552578,-0.746963,-0.608701,-0.507027,-0.508744,0.522878,-0.201303,-0.060793,-0.151571,-0.140800,-0.435182,0.346505,-0.325723,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,0.562626,1.552578,0.506520,0.361136,-0.896204,-0.972680,-0.066481,-0.201303,-0.060793,-0.151571,1.995173,1.877892,-0.060175,-0.325723,1,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0.562626,1.552578,1.133261,1.524940,1.244269,1.138864,-1.068392,-0.201303,-0.060793,-0.151571,0.931770,0.640195,-1.029948,-0.325723,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,0.258446,0.543861,-1.164791,-1.190604,-0.623780,-0.877351,-1.422007,-0.201303,-0.060793,-0.151571,0.775927,0.356133,-1.029948,-0.325723,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
4,-0.958274,-1.473572,0.506520,0.361136,-0.961067,-0.828097,0.228198,-0.116392,-0.060793,-0.151571,-0.498323,-0.881564,0.909599,-0.325723,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


Treinamento

In [12]:
X = df.drop(columns=['IS_DELAYED'])
y = df['IS_DELAYED']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Distribuição das classes:\n{y.value_counts(normalize=True)}')
print(f'Treino: {X_train.shape}, Teste: {X_test.shape}')

Distribuição das classes:
IS_DELAYED
0    0.5
1    0.5
Name: proportion, dtype: float64
Treino: (1083096, 36), Teste: (270774, 36)


In [13]:
print('Treinando Regressão Logística...')
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
print('\nResultado: Regressão Logística')
print(classification_report(y_test, y_pred_lr))

Treinando Regressão Logística...

Resultado: Regressão Logística
              precision    recall  f1-score   support

           0       0.61      0.59      0.60    135387
           1       0.61      0.63      0.62    135387

    accuracy                           0.61    270774
   macro avg       0.61      0.61      0.61    270774
weighted avg       0.61      0.61      0.61    270774



In [14]:
print('Treinando XGBoost...')
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=10,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42,
    n_jobs=-1   
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
print('\nResultado:')
print(classification_report(y_test, y_pred_xgb))
print(f'ROC-AUC Score: {roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1]):.4f}')

Treinando XGBoost...

Resultado:
              precision    recall  f1-score   support

           0       0.64      0.64      0.64    135387
           1       0.64      0.65      0.64    135387

    accuracy                           0.64    270774
   macro avg       0.64      0.64      0.64    270774
weighted avg       0.64      0.64      0.64    270774

ROC-AUC Score: 0.7010


In [15]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    x = ['Previsto Pontual', 'Previsto Atrasado']
    y = ['Real Pontual', 'Real Atrasado']
    
    fig = ff.create_annotated_heatmap(
        z=cm, 
        x=x, 
        y=y, 
        annotation_text=cm, 
        colorscale='Blues'
    )
    fig.update_layout(
        title=title,
        xaxis_title='Predição',
        yaxis_title='Realidade',
        template='plotly_white'
    )
    return fig

In [16]:
fig_xgb = plot_confusion_matrix(y_test, y_pred_xgb, 'Matriz de Confusão: XGBoost')
fig_xgb.show()

In [17]:
def plot_roc_curve(y_true, y_probs, title):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    auc_score = roc_auc_score(y_true, y_probs)

    fig = go.Figure()

    fig.add_trace(go.Scatter(
            x=fpr, 
            y=tpr,
            mode='lines',
            name=f'XGBoost (AUC = {auc_score:.4f})',
            line=dict(color='darkblue', width=3)
        )
    )

    fig.add_trace(go.Scatter(
            x=[0, 1], 
            y=[0, 1],
            mode='lines',
            name='Predição Aleatória',
            line=dict(color='red', dash='dash')
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title='Taxa de Falso Positivo (1 - Especificidade)',
        yaxis_title='Taxa de Verdadeiro Positivo (Sensibilidade)',
        height=600,
        template='plotly_white'
    )
    
    fig.show()

    return auc_score

In [18]:
y_probs_xgb = xgb_model.predict_proba(X_test)[:, 1]
plot_roc_curve(y_test, y_probs_xgb, 'Curva ROC')

0.7010272108692415

Que características aumentam a chance de atraso em um voo?

df_importances = pd.DataFrame({
    'Variável': features,
    'Importância': xgb_model.feature_importances_
}).sort_values(by='Importância', ascending=False)

fig = px.bar(
    df_importances,
    x='Importância',
    y='Variável',
    orientation='h',
    title='Importância das Variáveis Explanatórias'
)
fig.show()